Samajhna pehle:

- Scheduler (cron) vs Orchestrator: cron sirf “is waqt yeh command chalao” karta hai. Orchestrator jaanta hai ki kaunsa step kis par depend karta hai, fail hone par retry karta hai, aage ke steps rokta hai, sab log karta hai, aur kisi ek step se dobara shuru kar sakta hai.
- DAG (Directed Acyclic Graph): steps ka nakshaa, jisme teer dependency dikhate hain. Acyclic isliye ki A ko B chahiye aur B ko A, to koi pehle chal hi nahi sakta.
- Idempotent task: dobara chalane par wahi result de aur kuch bigaade nahi. Retries tabhi safe hain jab task idempotent ho. Iske liye output aam taur par temp file mein likh kar rename karte hain, taaki half-likhi file na bache.
- Data hand-off: tasks ek doosre ko file/artifact path dete hain, bade in-memory objects nahi.
- Schedule aur backfill: pipeline ko ek as_of date milti hai, aur purani dates ke liye wahi pipeline dobara chalayi ja sakti hai.
- Label latency (tumhare project ki khaas baat): tumhara label “cutoff ke baad ke 90 din” par bana hai. Aaj retrain karo to sabse naya cutoff aaj se 90 din pehle ka hoga, kyunki uske baad ka outcome abhi pata hi nahi. Scheduled retraining mein yeh delay design ka hissa hai.

### Tools ka naksha:

| Tool | Kaisa hai | Is project ke liye |
| :--- | :--- | :--- |
| **cron / GitHub Actions schedule** | Sirf time par command | Kaafi hai agar ek script mein sab ho |
| **Makefile / DVC** | File dependencies se "jo badla wahi dobara chalao" | Chhote ML pipelines ke liye achha |
| **Prefect** | Python decorators (`@flow`, `@task`), retries, UI. Setup halka | Concept dikhane ke liye best |
| **Apache Airflow** | Industry standard: scheduler, web UI, metadata database. Setup bhaari, aur native Windows par nahi chalta (Linux/macOS/WSL) | Interview mein naam aur concepts aane chahiye |

Ek solo, mahine mein ek baar retrain hone wale project ke liye Airflow overkill hai. Interview mein yeh kehna sahi hai ki tumne concepts samjhe aur apni pipeline ko DAG ke roop mein design kiya, aur tool ka chunav scale par depend karta hai.

 rules:

1. Aaj kuch bhi install nahi karna. Airflow aur Prefect bahut dependencies pin karte hain, aur pichhli baar jaisa MLflow ke saath hua, ye numpy/pandas/scikit-learn ko badal sakte hain. Agar baad mein khud try karna ho to alag virtual environment mein.
2. Demo mein asli project files (app_artifacts/, data/processed/) ko haath nahi lagana. Sab kuch temp folder mein hoga.
3. Aaj ka DAG nakshaa hai. Asli steps ko script mein badalne ka kaam kal (Day 94) hai.

1: Apni pipeline ka audit (kaun sa step script hai, kaun sa notebook mein atka hai)

In [2]:
import sys
sys.path.append('../../')

import tempfile
from pathlib import Path

import pandas as pd

from src.pipeline.mini_dag import DAG, Task

EXPECTED = {
    'clean':    ['src/cleaning/clean_data.py'],
    'features': ['src/features/build_features.py', 'src/features/rfm.py'],
    'split':    [],                                    # notebooks mein
    'tune/CV':  ['src/models/search_spaces.py', 'src/models/cv_compare.py', 'src/tracking/experiments.py'],
    'evaluate': ['src/models/evaluate.py', 'src/models/bootstrap.py'],
    'explain':  ['src/models/explain.py'],
    'export':   ['src/app/export_artifacts.py'],
    'serve':    ['src/app/service.py', 'src/app/app.py'],
}

def has_main(path):
    p = Path(path)
    return p.exists() and '__main__' in p.read_text()

rows = [{'step': step,
         'files': ', '.join(files) or '(koi script nahi)',
         'all_exist': all(Path(f).exists() for f in files) if files else False,
         'runnable_from_cli': any(has_main(f) for f in files)} for step, files in EXPECTED.items()]
audit = pd.DataFrame(rows).set_index('step')
print(audit.to_string())

                                                                                       files  all_exist  runnable_from_cli
step                                                                                                                      
clean                                                             src/cleaning/clean_data.py      False              False
features                                 src/features/build_features.py, src/features/rfm.py      False              False
split                                                                      (koi script nahi)      False              False
tune/CV   src/models/search_spaces.py, src/models/cv_compare.py, src/tracking/experiments.py      False              False
evaluate                                     src/models/evaluate.py, src/models/bootstrap.py      False              False
explain                                                                src/models/explain.py      False              False
export          

yeh table docs/pipeline_dag.md ke “Today” column ka asli saboot banega. Jo cheezein tumhe False dikhein, wahi pipeline ke gap hain.

 2: Stub tasks se orchestration ke concepts dekho (temp folder, asli data nahi)

In [3]:
tmp = Path(tempfile.mkdtemp())
raw, cleaned, feats, split, model, report = [tmp / n for n in
    ['raw.txt', 'cleaned.txt', 'feats.txt', 'split.txt', 'model.txt', 'report.txt']]
raw.write_text('A1,10\nA2,20\nA3,30\n')

STATE = {'attempts': 0, 'broken': False}

def clean():
    rows = [r for r in raw.read_text().splitlines() if not r.startswith('C')]   # 'C...' = cancellation
    cleaned.write_text('\n'.join(rows))

def build_features():
    feats.write_text('\n'.join(f'{r},feat' for r in cleaned.read_text().splitlines()))

def make_split():
    split.write_text(f"train:{len(feats.read_text().splitlines()) - 1} test:1")

def train():                                           # pehle 2 attempts "tracking server busy" se fail
    STATE['attempts'] += 1
    if STATE['broken'] or STATE['attempts'] <= 2:
        raise ConnectionError('tracking server busy')
    model.write_text('model:' + split.read_text())

def evaluate():
    report.write_text('report for ' + model.read_text())

dag = DAG([
    Task('clean', clean, inputs=[raw], outputs=[cleaned]),
    Task('features', build_features, deps=['clean'], inputs=[cleaned], outputs=[feats]),
    Task('split', make_split, deps=['features'], inputs=[feats], outputs=[split]),
    Task('train', train, deps=['split'], inputs=[split], outputs=[model], retries=2, retry_delay=0.1),
    Task('evaluate', evaluate, deps=['train'], inputs=[model], outputs=[report]),
], state_dir=tmp / '.state')

print('Levels:', dag.levels())

Levels: [['clean'], ['features'], ['split'], ['train'], ['evaluate']]


3: Chhe scenarios, har ek se ek concept

In [4]:
print('--- Run 1: pehli baar (train do baar fail, retries se pass) ---')
print(dag.run())

print('\n--- Run 2: kuch nahi badla ---')
print(dag.run())

print('\n--- Run 3: raw mein sirf cancellation row jodi (clean ise hata dega) ---')
raw.write_text(raw.read_text() + 'C9,-5\n')
print(dag.run())

print('\n--- Run 4: raw mein asli naya row ---')
raw.write_text(raw.read_text() + 'A4,40\n')
print(dag.run())

print('\n--- Run 5: naya data, lekin train toot gaya ---')
STATE['broken'] = True
raw.write_text(raw.read_text() + 'A5,50\n')
print(dag.run())

print('\n--- Run 6: train theek hua, wahin se aage ---')
STATE['broken'] = False
print(dag.run())

--- Run 1: pehli baar (train do baar fail, retries se pass) ---
[clean] success in 0.00s (attempt 1)
[features] success in 0.00s (attempt 1)
[split] success in 0.00s (attempt 1)
[train] attempt 1 failed: ConnectionError('tracking server busy')
[train] attempt 2 failed: ConnectionError('tracking server busy')
[train] success in 0.00s (attempt 3)
[evaluate] success in 0.00s (attempt 1)
{'clean': 'success', 'features': 'success', 'split': 'success', 'train': 'success', 'evaluate': 'success'}

--- Run 2: kuch nahi badla ---
[clean] cached (inputs unchanged, outputs present)
[features] cached (inputs unchanged, outputs present)
[split] cached (inputs unchanged, outputs present)
[train] cached (inputs unchanged, outputs present)
[evaluate] cached (inputs unchanged, outputs present)
{'clean': 'cached', 'features': 'cached', 'split': 'cached', 'train': 'cached', 'evaluate': 'cached'}

--- Run 3: raw mein sirf cancellation row jodi (clean ise hata dega) ---
[clean] success in 0.00s (attempt 1)


Kya dikhna chahiye:

- Run 1: train ke do attempts fail (log mein dikhega), teesre par success. Yeh retry hai.
- Run 2: sab cached. Inputs nahi badle, to kaam dobara nahi hota.
- Run 3: sirf clean chalta hai, baaki cached. Cancellation row hat gayi, to cleaned.txt bilkul same rahi, aur downstream ko pata hi nahi chalta ki raw badla. Isko early cutoff kehte hain (Day 34 ka cancellation filtering yaad karo).
- Run 4: asli naya row aaya, to poori chain dobara chali.
- Run 5: train ke teeno attempts fail, aur evaluate ko skipped_upstream mila. Agar evaluate chalta, to wo purani model.txt par report bana deta (stale output ka khatra).
- Run 6: clean, features, split cached. Sirf train aur evaluate dobara chale. Failure ke baad wahin se resume, jo orchestrator ka sabse kaam ka feature hai.

4: Apne asli project ke steps ek table mein

In [5]:
steps = pd.DataFrame([
    ['clean',     'clean_transactions()',                  'raw CSV',                          'step4_datatypes_fixed.csv',                  'function (CLI audit se dekho)'],
    ['features',  'build_customer_dataset(as_of=?)',       'step4_datatypes_fixed.csv',        'customer_ml_dataset_v2.csv',                 'script, cutoff implicit'],
    ['split',     '(notebooks 67 aur 83, aur export mein dobara)', 'customer_ml_dataset_v2.csv', 'X_train_v2, X_test_v2, y_train, y_test',   'NOTEBOOK ONLY, duplicate'],
    ['tune/CV',   'search_spaces, experiments',            'X_train_v2, y_train',              'MLflow runs, reports/*.csv',                 'library hai, driver notebook'],
    ['select',    'final_model_choice_day84.md',           'CV results',                       'pre-registered choice',                      'MANUAL BY DESIGN'],
    ['finalize',  '(notebook 86)',                         'splits + choice',                  'final_model.joblib, final_model_card.json',  'NOTEBOOK ONLY'],
    ['export',    'export_artifacts.py',                   'model + card + v2 csv',            'app_artifacts/',                             'script, input notebook se'],
    ['serve',     'src/app/app.py',                        'app_artifacts/',                   '-',                                          'script'],
], columns=['step', 'code', 'reads', 'writes', 'today'])
print(steps.to_string(index=False))

    step                                          code                      reads                                    writes                         today
   clean                          clean_transactions()                    raw CSV                 step4_datatypes_fixed.csv function (CLI audit se dekho)
features               build_customer_dataset(as_of=?)  step4_datatypes_fixed.csv                customer_ml_dataset_v2.csv       script, cutoff implicit
   split (notebooks 67 aur 83, aur export mein dobara) customer_ml_dataset_v2.csv    X_train_v2, X_test_v2, y_train, y_test      NOTEBOOK ONLY, duplicate
 tune/CV                    search_spaces, experiments        X_train_v2, y_train                MLflow runs, reports/*.csv  library hai, driver notebook
  select                   final_model_choice_day84.md                 CV results                     pre-registered choice              MANUAL BY DESIGN
finalize                                 (notebook 86)            splits + c

today column ko cell 1 ke audit se mila kar sahi karo. Mere andaaze se jo dikhta hai wo yeh hai, par asli saboot tumhara audit hai.